# Лабораторная работа: ансамблевые методы машинного обучения
(Градиентный бустинг, XGBoost, LightGBM)

*ML-3.2 Способен комплексно применять методы байесовской классификации и ансамблевые методы МО (бэггинг, бустинг, стэкинг моделей), а также производные от них (случайный лес, градиентный бустинг на деревьях), выдерживая баланс между вероятностной интерпретацией данных и комплексированием алгоритмов*

Сделайте копию этого файла (Файл - Сохранить копию на диске), переименуйте её, добавив в название вашу фамилию. Например, Иванова_Ансамбли_XGBoost.ipynb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits, load_wine, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score
)

# Для XGBoost и LightGBM
try:
    import xgboost as xgb
    print("XGBoost установлен")
except ImportError:
    print("XGBoost не установлен. Установите: pip install xgboost")

try:
    import lightgbm as lgb
    print("LightGBM установлен")
except ImportError:
    print("LightGBM не установлен. Установите: pip install lightgbm")

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("\nБиблиотеки загружены.")

# Часть 1. Введение в ансамблевые методы

Ансамблевые методы — это техники машинного обучения, которые комбинируют несколько моделей (базовых алгоритмов) для получения более мощного и устойчивого решения.

## 1.1. Основные подходы к ансамблированию

| Метод | Принцип | Примеры |
|-------|---------|---------|
| **Бэггинг (Bagging)** | Параллельное обучение моделей на разных подвыборках | Random Forest |
| **Бустинг (Boosting)** | Последовательное обучение, каждая новая модель исправляет ошибки предыдущих | AdaBoost, Gradient Boosting, XGBoost |
| **Стэкинг (Stacking)** | Обучение мета-модели на предсказаниях базовых моделей | StackingClassifier |
| **Голосование (Voting)** | Простое или взвешенное голосование | VotingClassifier |

## 1.2. Бустинг: ключевая идея

**Градиентный бустинг** — это метод, который:
1. Обучает базовую модель (обычно дерево)
2. Вычисляет ошибки (градиенты) на обучающей выборке
3. Обучает следующую модель на этих ошибках
4. Повторяет шаги 2-3 N раз
5. Итоговый прогноз — взвешенная сумма всех моделей

**Преимущества бустинга:**
- Высокая точность
- Работает с различными функциями потерь
- Устойчив к переобучению (при правильной настройке)

**Недостатки:**
- Чувствительность к выбросам
- Склонность к переобучению при неправильной настройке
- Медленнее обучается, чем случайный лес

**Вопросы для размышления (ответьте письменно):**

1. В чем ключевое отличие бустинга от бэггинга?
2. Почему бустинг часто показывает лучшее качество, чем случайный лес?
3. Каковы риски использования бустинга на зашумленных данных?

In [ ]:
# Загружаем датасет Digits
digits = load_digits()
X = digits.data
y = digits.target

print(f"Размерность данных: {X.shape}")
print(f"Количество классов: {len(np.unique(y))}")

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")

# Часть 2. Градиентный бустинг (GradientBoostingClassifier)

GradientBoostingClassifier из scikit-learn — базовая реализация градиентного бустинга.

**Ключевые параметры:**
- `n_estimators` — число деревьев (этапов бустинга)
- `learning_rate` — шаг обучения (скорость, с которой модель "учится")
- `max_depth` — максимальная глубина деревьев
- `subsample` — доля выборки для обучения каждого дерева
- `min_samples_split` — минимальное число объектов для разбиения

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Базовая модель
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print("=== Градиентный бустинг (GradientBoostingClassifier) ===")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_gb):.4f}")
print(f"F1-score (macro): {f1_score(y_test, y_pred_gb, average='macro'):.4f}")
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_gb))

## 2.1. Влияние learning_rate

**Практика.**

Обучите модели с разными learning_rate (0.01, 0.05, 0.1, 0.3, 0.5).

**Вопросы для размышления:**
- Как learning_rate влияет на качество?
- Как learning_rate влияет на время обучения?
- Почему маленький learning_rate требует большего числа деревьев?
- Какая зависимость между learning_rate и n_estimators?

In [ ]:
# Разместите здесь ваш код

# Анализ влияния learning_rate
learning_rates = [0.01, 0.05, 0.1, 0.3, 0.5]
scores = []
times = []

for lr in learning_rates:
    start = time.time()
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42)
    gb.fit(X_train, y_train)
    times.append(time.time() - start)
    scores.append(gb.score(X_test, y_test))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(learning_rates, scores, 'bo-', linewidth=2)
axes[0].set_xlabel('learning_rate')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Влияние learning_rate на качество')
axes[0].grid(True)

axes[1].plot(learning_rates, times, 'ro-', linewidth=2)
axes[1].set_xlabel('learning_rate')
axes[1].set_ylabel('Время обучения (сек)')
axes[1].set_title('Влияние learning_rate на время обучения')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("\nРезультаты:")
for lr, score, t in zip(learning_rates, scores, times):
    print(f"learning_rate={lr:.2f}: Accuracy={score:.4f}, Time={t:.2f}с")

## 2.2. GridSearchCV для градиентного бустинга

**Практика.**

Подберите оптимальные параметры для GradientBoostingClassifier.

**Рекомендуемые параметры:**
- `n_estimators`: [50, 100, 150]
- `learning_rate`: [0.05, 0.1, 0.2]
- `max_depth`: [3, 5, 7]
- `subsample`: [0.8, 1.0]

**Вопросы для размышления:**
- Какие параметры оказались оптимальными?
- Как изменилось качество по сравнению с параметрами по умолчанию?
- Почему в градиентном бустинге важно использовать subsample?

In [ ]:
# Разместите здесь ваш код

from sklearn.model_selection import GridSearchCV

# Упрощенная сетка для ускорения
param_grid_gb = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

gb = GradientBoostingClassifier(random_state=42)
grid_gb = GridSearchCV(gb, param_grid_gb, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)

start = time.time()
grid_gb.fit(X_train, y_train)
grid_time = time.time() - start

print(f"Лучшие параметры: {grid_gb.best_params_}")
print(f"Лучшая точность на CV: {grid_gb.best_score_:.4f}")
print(f"Время подбора: {grid_time:.2f} сек")

best_gb = grid_gb.best_estimator_
print(f"Точность на тесте: {best_gb.score(X_test, y_test):.4f}")

# Часть 3. XGBoost (eXtreme Gradient Boosting)

XGBoost — это оптимизированная реализация градиентного бустинга, которая стала стандартом в соревнованиях по машинному обучению.

**Ключевые особенности XGBoost:**
1. **Регуляризация** — L1 и L2 регуляризация для борьбы с переобучением
2. **Работа с пропусками** — встроенная обработка пропущенных значений
3. **Распараллеливание** — эффективное использование нескольких ядер
4. **Early stopping** — остановка обучения при отсутствии улучшений
5. **Встроенная кросс-валидация**

**Ключевые параметры:**
- `n_estimators` — число деревьев
- `learning_rate` — шаг обучения (эта)
- `max_depth` — максимальная глубина
- `subsample` — доля выборки для обучения
- `colsample_bytree` — доля признаков для каждого дерева
- `reg_alpha` — L1 регуляризация
- `reg_lambda` — L2 регуляризация
- `objective` — функция потерь ('multi:softmax' для классификации)
- `eval_metric` — метрика для оценки

In [ ]:
import xgboost as xgb

# Базовая модель XGBoost
xgb_clf = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

start = time.time()
xgb_clf.fit(X_train, y_train)
xgb_time = time.time() - start

y_pred_xgb = xgb_clf.predict(X_test)

print("=== XGBoost ===")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"F1-score (macro): {f1_score(y_test, y_pred_xgb, average='macro'):.4f}")
print(f"Время обучения: {xgb_time:.2f} сек")
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_xgb))

## 3.1. Визуализация важности признаков в XGBoost

XGBoost предоставляет несколько способов оценки важности признаков:
- `weight` — сколько раз признак использовался для разбиения
- `gain` — средний прирост качества при использовании признака
- `cover` — среднее покрытие признака (число объектов, которые через него проходят)

In [ ]:
# Важность признаков в XGBoost
importance_types = ['weight', 'gain', 'cover']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, imp_type in zip(axes, importance_types):
    importances = xgb_clf.get_booster().get_score(importance_type=imp_type)

    # Преобразуем в массив
    imp_array = np.zeros(64)
    for key, value in importances.items():
        idx = int(key.replace('f', ''))
        imp_array[idx] = value

    # Визуализация
    imp_img = imp_array.reshape(8, 8)
    ax.imshow(imp_img, cmap='hot', interpolation='nearest')
    ax.set_title(f'Важность: {imp_type}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3.2. GridSearchCV для XGBoost

**Практика.**

Подберите оптимальные параметры для XGBoost.

**Рекомендуемые параметры для перебора:**
- `n_estimators`: [50, 100, 150]
- `max_depth`: [3, 5, 7]
- `learning_rate`: [0.05, 0.1, 0.2]
- `subsample`: [0.8, 1.0]
- `colsample_bytree`: [0.8, 1.0]

**Вопросы для размышления:**
- Какие параметры оказались оптимальными для XGBoost?
- Чем отличается подбор параметров XGBoost от GradientBoostingClassifier?
- Почему в XGBoost важно настраивать регуляризацию (reg_alpha, reg_lambda)?

In [ ]:
# Разместите здесь ваш код

# GridSearchCV для XGBoost
param_grid_xgb = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_clf_search = xgb.XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

grid_xgb = GridSearchCV(xgb_clf_search, param_grid_xgb, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)

start = time.time()
grid_xgb.fit(X_train, y_train)
grid_time = time.time() - start

print(f"Лучшие параметры XGBoost: {grid_xgb.best_params_}")
print(f"Лучшая точность на CV: {grid_xgb.best_score_:.4f}")
print(f"Время подбора: {grid_time:.2f} сек")

best_xgb = grid_xgb.best_estimator_
print(f"Точность на тесте: {best_xgb.score(X_test, y_test):.4f}")

# Часть 4. LightGBM

LightGBM — это фреймворк градиентного бустинга, разработанный Microsoft. Он оптимизирован для скорости и эффективности работы с большими данными.

**Ключевые особенности LightGBM:**
1. **Leaf-wise рост деревьев** — рост в глубину, а не в ширину (в отличие от XGBoost)
2. **GOSS (Gradient-based One-Side Sampling)** — выборка важных градиентов
3. **EFB (Exclusive Feature Bundling)** — объединение взаимоисключающих признаков
4. **Встроенная работа с категориальными признаками** — без необходимости кодирования
5. **Быстрое обучение** — значительно быстрее XGBoost на больших данных

**Ключевые параметры:**
- `n_estimators` — число деревьев
- `learning_rate` — шаг обучения
- `num_leaves` — максимальное число листьев (аналог max_depth)
- `max_depth` — максимальная глубина (ограничение для leaf-wise)
- `subsample` — доля выборки для обучения
- `colsample_bytree` — доля признаков для каждого дерева
- `reg_alpha` — L1 регуляризация
- `reg_lambda` — L2 регуляризация
- `objective` — функция потерь ('multiclass' для классификации)

In [ ]:
import lightgbm as lgb

# Базовая модель LightGBM
lgb_clf = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

start = time.time()
lgb_clf.fit(X_train, y_train)
lgb_time = time.time() - start

y_pred_lgb = lgb_clf.predict(X_test)

print("=== LightGBM ===")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_lgb):.4f}")
print(f"F1-score (macro): {f1_score(y_test, y_pred_lgb, average='macro'):.4f}")
print(f"Время обучения: {lgb_time:.2f} сек")
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_lgb))

## 4.1. Сравнение LightGBM с XGBoost по скорости

**Практика.**

Сравните время обучения XGBoost и LightGBM при одинаковых параметрах.

**Вопросы для размышления:**
- Почему LightGBM обучается быстрее XGBoost?
- В чем компромисс между скоростью и качеством в LightGBM?
- Когда стоит использовать LightGBM вместо XGBoost, и наоборот?

In [ ]:
# Разместите здесь ваш код

# Сравнение скорости XGBoost vs LightGBM
n_estimators = 100
max_depth = 5
learning_rate = 0.1

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    learning_rate=learning_rate,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

start = time.time()
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start
xgb_acc = xgb_model.score(X_test, y_test)

# LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=n_estimators,
    num_leaves=2**max_depth,
    max_depth=max_depth,
    learning_rate=learning_rate,
    random_state=42,
    verbose=-1
)

start = time.time()
lgb_model.fit(X_train, y_train)
lgb_time = time.time() - start
lgb_acc = lgb_model.score(X_test, y_test)

print("=== Сравнение скорости ===")
print(f"XGBoost: Time={xgb_time:.2f}с, Accuracy={xgb_acc:.4f}")
print(f"LightGBM: Time={lgb_time:.2f}с, Accuracy={lgb_acc:.4f}")
print(f"LightGBM быстрее в {xgb_time/lgb_time:.1f} раз")

## 4.2. GridSearchCV для LightGBM

**Практика.**

Подберите оптимальные параметры для LightGBM.

**Рекомендуемые параметры для перебора:**
- `n_estimators`: [50, 100, 150]
- `num_leaves`: [15, 31, 63]
- `learning_rate`: [0.05, 0.1, 0.2]
- `subsample`: [0.8, 1.0]
- `colsample_bytree`: [0.8, 1.0]

**Вопросы для размышления:**
- Чем отличается настройка num_leaves от max_depth?
- Почему в LightGBM важно ограничивать num_leaves?
- Какие параметры LightGBM наиболее критичны для качества?

In [ ]:
# Разместите здесь ваш код

# GridSearchCV для LightGBM
param_grid_lgb = {
    'n_estimators': [50, 100],
    'num_leaves': [15, 31],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

lgb_clf_search = lgb.LGBMClassifier(
    random_state=42,
    verbose=-1
)

grid_lgb = GridSearchCV(lgb_clf_search, param_grid_lgb, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)

start = time.time()
grid_lgb.fit(X_train, y_train)
grid_time = time.time() - start

print(f"Лучшие параметры LightGBM: {grid_lgb.best_params_}")
print(f"Лучшая точность на CV: {grid_lgb.best_score_:.4f}")
print(f"Время подбора: {grid_time:.2f} сек")

best_lgb = grid_lgb.best_estimator_
print(f"Точность на тесте: {best_lgb.score(X_test, y_test):.4f}")

# Часть 5. Стэкинг и голосование

## 5.1. Voting Classifier (голосование)

VotingClassifier комбинирует предсказания нескольких моделей через голосование (hard) или усреднение вероятностей (soft).

In [ ]:
from sklearn.ensemble import VotingClassifier

# Создаем базовые модели
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

# Voting Classifier (hard voting)
voting_hard = VotingClassifier(
    estimators=[
        ('lr', model_lr),
        ('rf', model_rf),
        ('gb', model_gb)
    ],
    voting='hard'
)

# Voting Classifier (soft voting)
voting_soft = VotingClassifier(
    estimators=[
        ('lr', model_lr),
        ('rf', model_rf),
        ('gb', model_gb)
    ],
    voting='soft'
)

# Обучение
voting_hard.fit(X_train, y_train)
voting_soft.fit(X_train, y_train)

y_pred_voting_hard = voting_hard.predict(X_test)
y_pred_voting_soft = voting_soft.predict(X_test)

print("=== Voting Classifier ===")
print(f"Hard Voting Accuracy: {accuracy_score(y_test, y_pred_voting_hard):.4f}")
print(f"Soft Voting Accuracy: {accuracy_score(y_test, y_pred_voting_soft):.4f}")

## 5.2. Stacking Classifier (стэкинг)

StackingClassifier обучает мета-модель на предсказаниях базовых моделей. Это позволяет комбинировать сильные стороны разных алгоритмов.

In [ ]:
from sklearn.ensemble import StackingClassifier

# Базовые модели
base_models = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

# Мета-модель
meta_model = LogisticRegression(max_iter=1000, random_state=42)

# Stacking
stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

start = time.time()
stacking.fit(X_train, y_train)
stacking_time = time.time() - start

y_pred_stacking = stacking.predict(X_test)

print("=== Stacking Classifier ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_stacking):.4f}")
print(f"Время обучения: {stacking_time:.2f} сек")
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_stacking))

# Часть 6. Сравнение всех методов

**Практика.**

Сравните все изученные методы:
1. Random Forest (из предыдущей работы)
2. GradientBoostingClassifier
3. XGBoost
4. LightGBM
5. Voting Classifier (soft)
6. Stacking Classifier

**Вопросы для размышления:**
- Какой метод показал лучшее качество? Почему?
- Какой метод быстрее обучается?
- Какой метод лучше интерпретируется?
- В каких задачах вы бы использовали стэкинг?
- Почему ансамбли часто работают лучше одиночных моделей?

In [ ]:
# Разместите здесь ваш код

# Сравнение всех методов
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    'LightGBM': lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'Voting (soft)': voting_soft,
    'Stacking': stacking
}

results = []

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    results.append({
        'Модель': name,
        'Accuracy': acc,
        'F1 (macro)': f1,
        'Время обучения (с)': train_time,
        'Интерпретируемость': 'Средняя' if name in ['Random Forest', 'Gradient Boosting'] else 'Низкая'
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Часть 7. Работа с категориальными признаками в LightGBM

LightGBM имеет встроенную поддержку категориальных признаков, что является одним из его ключевых преимуществ.

**Практика.**

Создайте пример с категориальными данными и обучите LightGBM с указанием категориальных признаков.

In [ ]:
# Разместите здесь ваш код

# Пример с категориальными данными
from sklearn.datasets import make_classification
import pandas as pd

# Создаем данные с категориальными признаками
X_cat, y_cat = make_classification(n_samples=500, n_features=5, n_informative=3, random_state=42)
df_cat = pd.DataFrame(X_cat, columns=[f'feature_{i}' for i in range(5)])

# Добавляем категориальные признаки
df_cat['cat_feature'] = np.random.choice(['A', 'B', 'C', 'D'], size=500)
df_cat['cat_feature2'] = np.random.choice(['X', 'Y', 'Z'], size=500)

print("Данные с категориальными признаками:")
print(df_cat.head())

# Преобразование категорий в числовой формат для LightGBM
for col in ['cat_feature', 'cat_feature2']:
    df_cat[col] = df_cat[col].astype('category')

# Разделение
X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    df_cat, y_cat, test_size=0.3, random_state=42
)

# Обучение LightGBM с категориальными признаками
lgb_cat = lgb.LGBMClassifier(
    n_estimators=50,
    categorical_feature=['cat_feature', 'cat_feature2'],
    random_state=42,
    verbose=-1
)

lgb_cat.fit(X_train_cat, y_train_cat)
print(f"\nLightGBM с категориями: Accuracy = {lgb_cat.score(X_test_cat, y_test_cat):.4f}")

# Часть 8. Интерпретируемость ансамблей

Ансамблевые методы, особенно бустинг, часто критикуют за сложность интерпретации. Однако существуют инструменты, которые помогают понять, как работает модель.

**Инструменты интерпретации:**
1. **SHAP (SHapley Additive exPlanations)** — объяснение предсказаний через вклад каждого признака
2. **LIME (Local Interpretable Model-agnostic Explanations)** — локальные объяснения
3. **Feature Importance** — глобальная важность признаков
4. **Partial Dependence Plots (PDP)** — зависимость предсказания от одного признака

**Вопросы для размышления:**
- Почему ансамбли считаются "черными ящиками"?
- В каких случаях интерпретируемость критична?
- Как можно объяснить предсказание XGBoost бизнес-пользователю?

In [ ]:
# SHAP для интерпретации (требуется установка: pip install shap)
try:
    import shap
    print("SHAP установлен")

    # SHAP на небольшой выборке
    explainer = shap.TreeExplainer(best_xgb)
    shap_values = explainer.shap_values(X_test[:10])

    # Визуализация
    shap.summary_plot(shap_values, X_test[:10], feature_names=digits.feature_names, show=False)
    plt.title('SHAP: вклад признаков в предсказание')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("SHAP не установлен. Установите: pip install shap")

# Итоговые вопросы для размышления

**Ответьте письменно:**

1. В чем ключевое отличие градиентного бустинга от случайного леса? Когда какой метод предпочтительнее?

2. Почему XGBoost стал "стандартом де-факто" в соревнованиях по машинному обучению?

3. В чем преимущества LightGBM перед XGBoost? Когда стоит использовать LightGBM?

4. Что такое стэкинг и в каких случаях он эффективен?

5. Как регуляризация (L1, L2) в XGBoost помогает бороться с переобучением?

6. Как выбрать между GradientBoostingClassifier, XGBoost и LightGBM в зависимости от задачи?

7. В чем компромисс между интерпретируемостью и качеством при использовании ансамблей?

8. Какие инструменты интерпретации (SHAP, LIME) вы бы использовали для объяснения предсказаний бустинга?

**Эссе (7-10 предложений):**

На основе проведенных экспериментов опишите:
1. Какой ансамблевый метод показал лучшее качество на датасете digits и почему
2. Сравните методы по скорости обучения, качеству и интерпретируемости
3. В каких реальных задачах вы бы использовали каждый из методов
4. Как вы будете выбирать между бэггингом и бустингом в зависимости от задачи

**Ваше эссе:**

# Дополнительное задание

**Практика.**

Примените изученные методы к датасету load_wine (3 класса, 13 признаков).

**Задачи:**
1. Загрузите данные, выполните стандартизацию
2. Обучите XGBoost и LightGBM с подбором параметров
3. Сравните качество с GradientBoostingClassifier
4. Используйте Voting и Stacking
5. Сделайте выводы о применимости разных методов

**Вопросы:**
- Как изменилось качество на Wine по сравнению с Digits?
- Почему на Wine все методы показывают высокое качество?
- Как число классов и признаков влияет на выбор метода?

In [ ]:
from sklearn.datasets import load_wine

# Разместите здесь ваш код

# Загрузка и подготовка данных
wine = load_wine()
X_w, y_w = wine.data, wine.target

print(f"Размерность Wine: {X_w.shape}")
print(f"Количество классов: {len(np.unique(y_w))}")

# Стандартизация
scaler = StandardScaler()
X_w_scaled = scaler.fit_transform(X_w)

# Разделение
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_w_scaled, y_w, test_size=0.3, random_state=42)

# Ваш код для сравнения методов на Wine